# Case 5 — Three-Well Staggered Startup, Multi-Rate Production
## POD Analysis: Individual Drawdown, Interference, Buildup


In [ ]:
import numpy as np
import h5py, json
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.interpolate import UnivariateSpline
from matplotlib.ticker import LogLocator, LogFormatter
from scipy.special import expi
from pathlib import Path
import matplotlib.patches as patches
import matplotlib.ticker as ticker
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
import warnings
from matplotlib.ticker import NullLocator

from matplotlib.patches import Patch
from scipy.interpolate import interp1d
import matplotlib.patches as mpatches
from matplotlib.lines   import Line2D
import matplotlib.lines as mlines
plt.rcParams.update({
    # 1. True LaTeX Font Rendering
    'text.usetex': True,
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman'], # Native LaTeX font
    
    # Optional
    'text.latex.preamble': r'\usepackage{amsmath} \usepackage{amssymb} \usepackage{bm}',

    # 2. Font Sizes 
    'font.size': 14,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'legend.fontsize': 14,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,

    # 3. Ticks: Inward-facing and mirrored on top/right
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'xtick.top': True,
    'ytick.right': True,
    'xtick.minor.visible': False,
    'ytick.minor.visible': False,

    # 4. Line and Spine Weights
    'axes.linewidth': 1,      # Frame thickness
    'lines.linewidth': 1.5,     # Data line thickness
    'lines.markersize': 6,      # Default marker size

    # 5. Grid (Keep it subtle if used)
    'axes.grid': False,
    'grid.alpha': 0.20,
    'grid.linestyle': '--',
    'grid.linewidth': 0.5,

    # 6. Saving and DPI
    'figure.dpi': 150,          # For crisp screen preview in Jupyter/Spyder
    'savefig.dpi': 600,         # High resolution for line art
    'savefig.bbox': 'tight',    # Prevents labels from getting cut off

    # legend
    # 7. Legend Formatting
    'legend.frameon': True,         
    'legend.framealpha': 1.0,       
    'legend.edgecolor': 'black',    
    'legend.fancybox': False,       
    'legend.handlelength': 1.5,     
    'legend.labelspacing': 0.4,     
    'legend.handletextpad': 0.5,    
    'legend.borderaxespad': 0.5,   

    # axes padding
    'xtick.major.pad': 5.0,  # Default is usually 3.5
    'ytick.major.pad': 5.0, 
})

C1,C2,C3,C4 = '#1a5e9e','#0d6e56','#9b2a1a','#b07318'
GR = '#555555'
WC = ['#1a5e9e','#9b2a1a','#0d6e56']   # W1 blue, W2 red, W3 green

PROJECT = Path('..')
DATA    = PROJECT/'data'/'01_synthetic'/'case5_multiwell'
FIGS = PROJECT/'figures'
FIGS.mkdir(parents=True, exist_ok=True)
def sf(n): plt.savefig(FIGS/n,dpi=300,bbox_inches='tight'); print(f'  {n}')
print('Ready.')

---
## 1 · Load Simulation Data

In [ ]:
with h5py.File(DATA/'snapshots_psia.mat','r') as f:
    _raw   = np.array(f['X_psia'])
    X_psia = _raw.T if _raw.shape[0]<_raw.shape[1] else _raw

with h5py.File(DATA/'well_data.mat','r') as f:
    bhp_raw  = np.array(f['bhp_psia'])
    dp_raw   = np.array(f['delta_p_psi'])
    q_raw    = np.array(f['rate_STBd'])
    t_raw    = np.array(f['time_hr']).flatten()
    qtot_raw = np.array(f['total_rate']).flatten()
    Np_raw   = np.array(f['cum_prod_STB']).flatten()

# fix h5py orientation
for arr in ['bhp_raw','dp_raw','q_raw']:
    v = eval(arr)
    if v.shape[0] > v.shape[1]:
        exec(f'{arr} = v.T')
# safer explicit fix:
if bhp_raw.shape[0] > bhp_raw.shape[1]: bhp_raw = bhp_raw.T
if dp_raw.shape[0]  > dp_raw.shape[1]:  dp_raw  = dp_raw.T
if q_raw.shape[0]   > q_raw.shape[1]:   q_raw   = q_raw.T

with h5py.File(DATA/'rock_field.mat','r') as f:
    perm_mD = np.array(f['perm_mD']).flatten()

with open(DATA/'params.json') as f:
    P = json.load(f)

n_t    = min(len(t_raw), X_psia.shape[1])
t_raw  = t_raw[:n_t]; qtot_raw=qtot_raw[:n_t]; Np_raw=Np_raw[:n_t]
bhp_raw=bhp_raw[:,:n_t]; dp_raw=dp_raw[:,:n_t]; q_raw=q_raw[:,:n_t]
X_psia = X_psia[:,:n_t]

pi_   = P['p_init_psia']
valid = (t_raw > 0)
t    = t_raw[valid]
bhp  = bhp_raw[:, valid]   # (3, m)
dp   = dp_raw[:, valid]    # (3, m)
q    = q_raw[:, valid]     # (3, m)
qtot = qtot_raw[valid]
Np   = Np_raw[valid]
X    = X_psia[:, valid]

N, m  = X.shape
nx    = int(P['nx']); ny = int(P['ny'])
wc    = [int(P['wc_W1'])-1, int(P['wc_W2'])-1, int(P['wc_W3'])-1]

# Timescales
t_wbs   = P['t_wbs_end_hr']
t_int   = P['t_int_hr']
t_pss   = P['t_pss_start_hr']   # BDF onset
t_shut  = P['t_shutin_hr']

# Well startup and change times
t_W1s = P['t_W1_start_hr']
t_W2s = P['t_W2_start_hr']
t_W3s = P['t_W3_start_hr']
t_W1c = P['t_W1_change_hr']
t_W2c = P['t_W2_change_hr']
t_W3c = P['t_W3_change_hr']

# Period boundary times
t_ends = [P[f't_period{pp}_end_hr'] for pp in range(1,8)]

# Period masks
m_p = []
t_starts_all = [t[0], t_ends[0], t_ends[1], t_ends[2],
                t_ends[3], t_ends[4], t_ends[5]]
for pp in range(7):
    m_p.append((t > t_starts_all[pp]) & (t <= t_ends[pp]))

# Flow regime masks
m_w1only = t <= t_W2s
m_w1w2   = (t > t_W2s) & (t <= t_W3s)
m_all3   = (t > t_W3s) & (t < t_shut)
m_bu     = t >= t_shut
m_pre    = (t > 0)     & (t < t_int)
m_int    = (t >= t_int)& (t < t_pss)
m_bdf    = (t >= t_pss)& (t < t_shut)

print(f'X shape          : {X.shape}')
print(f'Time range       : {t[0]:.5f} – {t[-1]:.0f} hr')
print(f'BHP ranges:')
for wi in range(3):
    print(f'  W{wi+1}: {bhp[wi].min():.1f} – {bhp[wi].max():.1f} psia')
print(f'\nTimescale hierarchy:')
print(f'  t_WBS={t_wbs:.1f}  t_int={t_int:.0f}  t_BDF={t_pss:.0f}  '
      f't_shut={t_shut:.0f}  t_end={t[-1]:.0f} hr')
print(f'\nWell events:')
print(f'  W1 starts: {t_W1s:.3f} hr  |  W2: {t_W2s:.0f} hr  |  W3: {t_W3s:.0f} hr')
print(f'  W1 change: {t_W1c:.0f} hr  |  W2: {t_W2c:.0f} hr  |  W3: {t_W3c:.0f} hr')
print(f'\nSteps per period: '
      +' | '.join([f'P{pp+1}={m_p[pp].sum()}' for pp in range(7)]))

---
## 2 · Rate Schedule and BHP History

---
## 3 · Bourdet Diagnostics

Staggered startup: W1 alone (WBS→flat at $70.6\,q_{W1}\mu B_o/(kh)$→BDF), W1+W2 interference from $t\approx500$ hr, all-three interference from $t\approx1321$ hr (derivatives rise above individual flat levels).


In [ ]:
def bourdet(dp_arr, t_arr, L=0.2):
    n,lnt,d=len(dp_arr),np.log(t_arr),np.zeros(len(dp_arr))
    for i in range(1,n-1):
        jl=i-1
        while jl>0   and lnt[i]-lnt[jl]<L: jl-=1
        jr=i+1
        while jr<n-1 and lnt[jr]-lnt[i]<L: jr+=1
        dL,dR=lnt[i]-lnt[jl],lnt[jr]-lnt[i]
        if dL>0 and dR>0:
            d[i]=((dp_arr[i]-dp_arr[jl])/dL*dR+(dp_arr[jr]-dp_arr[i])/dR*dL)/(dL+dR)
        elif dL>0: d[i]=(dp_arr[i]-dp_arr[jl])/dL
        elif dR>0: d[i]=(dp_arr[jr]-dp_arr[i])/dR
    d[0]=(dp_arr[1]-dp_arr[0])/(lnt[1]-lnt[0]) if lnt[1]!=lnt[0] else 0
    d[-1]=(dp_arr[-1]-dp_arr[-2])/(lnt[-1]-lnt[-2]) if lnt[-1]!=lnt[-2] else 0
    return d

k=P['k_mD']; phi_=P['phi']; ct=P['ct_psi1']; mu=P['mu_cp']
kh=P['kh_mDft']; Bo=P['Bo']; h_ft=P['h_ft']; rw=P['rw_ft']

# Single-well flat levels
q_init = [P['q_P1_W1'], P['q_P1_W2'], P['q_P1_W3']]
dp_flat_sw = [70.6*q_init[wi]*mu*Bo/kh for wi in range(3)]
print('Single-well Bourdet flat levels:')
for wi in range(3):
    print(f'  W{wi+1} (q={q_init[wi]:.0f}): {dp_flat_sw[wi]:.4f} psi')

### Figure 2 — W1 Individual Bourdet (P1 only, $t<500$ hr)


In [ ]:
# W1 alone during P1 — cleanest diagnostic, no interference
t_p1_mask = (t >= 0.1) & (t <= t_W2s)
if t_p1_mask.sum() < 5:
    t_p1_mask = (t >= 0.1) & (t <= t_ends[0])

dp_W1_p1  = np.clip(dp[0, t_p1_mask], 0, None)
t_W1_p1   = t[t_p1_mask]
dpr_W1_p1 = bourdet(dp_W1_p1, t_W1_p1, L=0.2)
vs_W1_p1  = (dpr_W1_p1>1e-2)&(dp_W1_p1>1e-2)

fig, axes = plt.subplots(1,2,figsize=(12,5))

ax = axes[0]
ax.semilogx(t_W1_p1, bhp[0,t_p1_mask],
            color=WC[0], lw=2.0, label='W1 (alone)')
ax.axvline(t_wbs, color='green', lw=2, ls=':', alpha=0.8,
           label=f'$t_{{WBS}}={t_wbs:.1f}$ hr')
ax.axvspan(t[0],t_wbs,color='C3',alpha=0.15)
ax.axvspan(t_wbs,t_W2s,color='C2',alpha=0.12)
ax.set_xlabel('$t$ [hours]'); ax.set_ylabel('$p_{wf,W1}$ [psia]')
ax.set_title('(a)  W1 BHP — individual drawdown',loc='left',weight='normal')
ax.legend(fontsize=9,framealpha=0.93)

ax2 = axes[1]
ax2.axvspan(0.1,t_wbs,color='C3',alpha=0.15)
ax2.axvspan(t_wbs,t_W2s,color='C2',alpha=0.12)
ax2.loglog(t_W1_p1[vs_W1_p1], dpr_W1_p1[vs_W1_p1],
           color=WC[0], lw=2.0, label='W1 Bourdet')
ax2.axhline(dp_flat_sw[0], color='C1', lw=2.5, ls='--',
            label=rf"$\Delta p'_{{W1}}={dp_flat_sw[0]:.2f}$ psi")
ax2.axvline(t_wbs, color='green', lw=2, ls=':', alpha=0.8)
ax2.set_xlabel('$t$ [hours]')
ax2.set_ylabel(r"$\Delta p'$ [psi]")
ax2.set_title('(b)  W1 Bourdet — individual drawdown',
              loc='left',weight='normal')
ax2.set_xlim([0.1, t_W2s]); ax2.set_ylim([1,1000])
ax2.legend(loc='upper left',fontsize=9,framealpha=0.93,edgecolor='k')
plt.tight_layout()
plt.show()

### Figure 3 — W1 Full Pressure and Bourdet Derivative

W1's full history: interference from W2 ($t=500$ hr) and W3 ($t=1500$ hr), W1 rate change ($t=3000$ hr), BDF ($t\geq13212$ hr).


In [ ]:
# =============================================================================
#  Combined Figure — (a) Rate schedule  |  (b) W1 BHP  |  (c) W1 Bourdet Δp'
#  Insert this cell immediately after the fig3 cell.
#  Reuses all variables defined in fig3 and earlier cells; no reload needed.
# =============================================================================


FS_LABEL  = 14    # axis labels, panel letters
FS_TICK   = 12    # tick labels
FS_INSIDE = 11    # legend text, in-plot annotations
LW        = 3.0   # uniform data line weight

X_LO, X_HI = 0.1, 1e5   # shared x-axis limits (identical to fig3)

# ── Event-line x-axis-fraction helper ────────────────────────────────────────
# Recomputes axis fractions against the actual [X_LO, X_HI] log range so that
# text labels align correctly regardless of the original fig3 xlim.
def _xf(tv, dx=0.0):
    return (np.log10(max(tv, X_LO * 1.01)) - np.log10(X_LO)) / \
           (np.log10(X_HI) - np.log10(X_LO)) + dx

# Event definitions — (time, colour, label, x_nudge, y_top_frac, y_bot_frac)
# y_top used on panels (a)/(b); y_bot on panel (c) which has log y-scale
ev = [
    (t_W2s, WC[1],    r'PW2 starts',               -0.06, 0.20, 0.90),
    (t_W3s, WC[2],    r'PW3 starts',               0.06, 0.20, 0.90),
    (t_W1c, WC[0],    r'PW1 rate $\downarrow$',     0.06, 0.90, 0.90),
    (t_int, 'purple', r'$t_{\mathrm{int}}$',        -0.02, 0.68, 0.68),
    (t_pss, 'blue',   r'$t_{\mathrm{BDF}}$',        -0.02, 0.63, 0.68),
]

def _draw_vlines(ax, label_yf=None):
    """Draw event vlines on ax.  Pass label_yf to also add text labels."""
    for tv, col_, lbl, dx, yt, yb in ev:
        ax.axvline(tv, color=col_, lw=1.5, ls='--', alpha=0.75)
        if label_yf is not None:
            xf = _xf(tv, dx)
            if 0.02 < xf < 0.96:
                yf = yt if label_yf == 'top' else yb
                ax.text(xf, yf, lbl,
                        transform=ax.transAxes,
                        fontsize=FS_INSIDE - 1, ha='center', color=col_,
                        bbox=dict(fc='white', alpha=0.80, ec='none', pad=1.5))

# ── Figure and shared axes ────────────────────────────────────────────────────
fig, axes = plt.subplots(
    3, 1, figsize=(8, 6.5), sharex=True,
    gridspec_kw={'hspace': 0.2},
)
ax_q, ax_p, ax_d = axes

# =============================================================================
# Panel (a) — Per-well rate history (step plot, log x)
# =============================================================================
for wi in range(3):
    ax_q.step(t, q[wi], where='post',
              color=WC[wi], lw=LW, label=f'PW{wi+1}')

ax_q.set_xscale('log')        # propagates to ax_p and ax_d via sharex=True
_draw_vlines(ax_q, label_yf='top')   # event labels shown only on top panel

ax_q.set_ylabel(r'$q$ [STB/day]',  fontsize=FS_LABEL)
ax_q.set_ylim([-50, 1000])
ax_q.tick_params(labelbottom=False, labelsize=FS_TICK)
ax_q.xaxis.set_minor_locator(NullLocator())
ax_q.yaxis.set_minor_locator(NullLocator())
ax_q.legend(fontsize=FS_INSIDE, framealpha=0.93, ncol=3,
            loc='lower left', bbox_to_anchor=(0.05, 0.05), handlelength=1.2, labelspacing=0.3)
ax_q.set_title('(a)', loc='left', fontsize=FS_LABEL, fontweight='normal', pad=3)

# =============================================================================
# Panel (b) — W1 BHP full history (production periods only)
# =============================================================================
ax_p.plot(t_W1_full, bhp[0, t_prod_mask],
          color='blue', lw=LW)       # semilogx not needed — sharex already log

_draw_vlines(ax_p)                   # vlines only; labels already on (a)

# Retain the t_int / t_BDF labels since they sit at different y-positions
ax_p.text(_xf(t_int, -0.02), 0.68, r'$t_{\mathrm{int}}$',
          transform=ax_p.transAxes, fontsize=FS_INSIDE - 1, ha='center',
          color='purple',
          bbox=dict(fc='white', alpha=0.80, ec='none', pad=1.5))
ax_p.text(_xf(t_pss, -0.02), 0.40, r'$t_{\mathrm{BDF}}$',
          transform=ax_p.transAxes, fontsize=FS_INSIDE - 1, ha='center',
          color='blue',
          bbox=dict(fc='white', alpha=0.80, ec='none', pad=1.5))

ax_p.set_ylabel(r'$p_{wf,\,W1}$ [psia]', fontsize=FS_LABEL)
ax_p.set_ylim([3800, 4200])
ax_p.tick_params(labelbottom=False, labelsize=FS_TICK)
ax_p.xaxis.set_minor_locator(NullLocator())
ax_p.yaxis.set_minor_locator(NullLocator())
ax_p.set_title('(b)', loc='left', fontsize=FS_LABEL, fontweight='normal', pad=3)

# =============================================================================
# Panel (c) — W1 Bourdet derivative Δp' (colour-segmented by flow regime)
# =============================================================================
t_plot_c   = t_W1_full[vs_W1]
dpr_plot_c = dpr_W1_full[vs_W1]
cs_plot_c  = [col_seg[j] for j in range(len(t_W1_full)) if vs_W1[j]]

for j in range(len(t_plot_c) - 1):
    ax_d.loglog(t_plot_c[j:j+2], dpr_plot_c[j:j+2],
                color=cs_plot_c[j], lw=LW, alpha=0.9)

ax_d.axhline(dp_flat_sw[0], color=WC[0], lw=LW - 0.5, ls=':',
             label="W1 (radial)")

_draw_vlines(ax_d)     # vlines only on (c)

leg_patches = [
    Patch(fc=COL_ALONE, label='PW1 (rate 1)'),
    Patch(fc=COL_INT2,  label='PW2 interference'),
    Patch(fc=COL_INT3,  label='PW3 interference'),
    Patch(fc=COL_CHG,   label='PW1 (rate 2)'),
    Patch(fc=COL_BDF,   label='BDF'),
]
ref_lines, ref_labels = ax_d.get_legend_handles_labels()
ax_d.legend(
    handles=leg_patches + ref_lines,
    labels=[p.get_label() for p in leg_patches] + ref_labels,
    fontsize=FS_INSIDE, framealpha=0.93, loc='lower center', bbox_to_anchor=(0.35, 0.01),
    edgecolor='k', ncol=2, handlelength=1.2, labelspacing=0.3,
)

ax_d.set_xlabel(r'$t$ [hours]',             fontsize=FS_LABEL)
ax_d.set_ylabel(r"$\Delta p'_{W1}$ [psi]",  fontsize=FS_LABEL)
ax_d.set_xlim([X_LO, X_HI])
ax_d.set_ylim([0.1, 1e2])
ax_d.tick_params(labelsize=FS_TICK)
ax_d.xaxis.set_minor_locator(NullLocator())
ax_d.yaxis.set_minor_locator(NullLocator())
ax_d.set_title('(c)', loc='left', fontsize=FS_LABEL, fontweight='normal', pad=3)

# ── Save ──────────────────────────────────────────────────────────────────────
plt.tight_layout()
sf('ch4_6_q_pwf1_dp_case5.pdf')
plt.show()

---
## 4 · Pressure Field Snapshots

Six snapshots: W1 alone → W1+W2 → all three → interference → BDF with W1 change → buildup.


In [ ]:
dX = P['p_init_psia'] - X
def safe_snap(tv):
    idx = int(np.argmin(np.abs(t-tv)))
    return min(idx, m-1)

snaps = [
    (safe_snap(t_W2s*0.8),
     f'W1 only  $t={t_W2s*0.8:.0f}$ hr', WC[0]),
    (safe_snap(t_W3s*0.8),
     f'W1+W2    $t={t_W3s*0.8:.0f}$ hr', WC[1]),
    (safe_snap(t_int*1.5),
     f'Interference  $t={t_int*1.5:.0f}$ hr', '#9b2a1a'),
    (safe_snap(t_W1c*1.3),
     f'W1 changed  $t={t_W1c*1.3:.0f}$ hr', WC[0]),
    (safe_snap(t_pss*1.1),
     f'BDF  $t={t_pss*1.1:.0f}$ hr', WC[2]),
    (safe_snap((t_shut+t[-1])/2) if m_bu.any() else safe_snap(t[-1]),
     f'Buildup  $t={(t_shut+t[-1])/2:.0f}$ hr', '#555555'),
]

fig, axes = plt.subplots(2,3,figsize=(8.5,4))
for ax,(ki,ttl,col) in zip(axes.flat, snaps):
    F  = dX[:,ki].reshape(ny,nx)
    vm = max(F.max(),0.1)
    im = ax.imshow(F,origin='lower',cmap='hot_r',vmin=0,vmax=vm,aspect='auto')
    for wi,wci in enumerate(wc):
        ix_py=wci%nx; iy_py=wci//nx
        # dim inactive wells
        is_active = q[wi,ki] > 1.0
        mk = '+' if is_active else 'o'
        ms = 12 if is_active else 7
        ax.plot(ix_py,iy_py,mk,color=WC[wi],ms=ms,
                mew=1.5,mec='w',zorder=10)
    for lp in [0.25,0.50,0.75]:
        lv=vm*lp
        if lv>0.05:
            ax.contour(F,levels=[lv],colors=['w'],linewidths=0.5,alpha=0.4)
    ax.set_title(ttl,fontsize=12,color=col,fontweight='normal')
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.add_patch(plt.Rectangle((0,0),1,1,fill=False,ec=col,lw=2.5,
                               transform=ax.transAxes,clip_on=False))
    cb=plt.colorbar(im,ax=ax,shrink=0.85)
    cb.set_ticks([0,vm/2,vm])
    cb.ax.yaxis.set_major_formatter(plt.FormatStrFormatter('%.0f'))
    cb.set_label(r'$\Delta p$ [psi]',fontsize=12)
    #ax.text(0.97,0.03,f'max={vm:.0f}',transform=ax.transAxes,
           # fontsize=12,ha='right',va='bottom',color='white',
           # family='monospace',
           # bbox=dict(fc='k',alpha=0.6,pad=1.5,ec='none'))
plt.tight_layout()
sf('ch4_11_p_field_case5.pdf')
plt.show()

---
## 5 · POD Analysis

In [ ]:
X_mean = X.mean(axis=1,keepdims=True)
Xc     = X - X_mean
U, sv, Vt = np.linalg.svd(Xc, full_matrices=False)

sn  = sv/sv[0]
cum = np.cumsum(sv**2)/np.sum(sv**2)*100
r99  = int(np.searchsorted(cum,99.0))+1
r999 = int(np.searchsorted(cum,99.9))+1
r_sel = max(r999,2)

print(f'Snapshot matrix  : {X.shape}')
print(f'σ₁               : {sv[0]:.2f} psia')
print(f'σ₂/σ₁            : {sn[1]:.3e}')
print(f'r_99.0%%         : {r99}')
print(f'r_99.98%%         : {r999}  ← r_sel={r_sel}')
for i in range(min(8,len(sv))):
    print(f'  Mode {i+1}  σ={sv[i]:.2f}  σᵢ/σ₁={sn[i]:.2e}  cumE={cum[i]:.4f}%')

### 5.1 Figure 5 — Singular Value Spectrum

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

# --- Plot (a): Singular Value Decay ---
ax = axes[0]
ax.semilogy(range(1, 31), sn[:30], 'o', color=C1, ms=6.5, mec='k', mew=0.5)
ax.semilogy(range(1, 31), sn[:30], color='gray', alpha=0.3, lw=3, zorder=1)

ax.set_title('(a)', loc='left', fontweight='normal')
ax.set_xlabel('$r$')
ax.set_ylabel(r'$\sigma_r / \sigma_1$ [-]')

ax.set_xlim([0, 30])
ax.set_ylim([1e-5, 2])
ax.xaxis.set_minor_locator(NullLocator())
ax.yaxis.set_minor_locator(NullLocator())

# for idx, col in [(0, 'C3'), (1, 'C0'), (2, 'C5'), (3, 'C8')]:
#     ax.axvline(idx+1, color=col, ls=':', lw=1.5)
#     ax.text(idx+1, sn[min(idx, len(sn)-1)] * 0.2, f'$i={idx+1}$',
#             color=col, rotation=90, ha='right', va='center', fontweight='bold',
#             bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=1))

# --- Plot (b): Cumulative Energy ---
ax2 = axes[1]
ax2.plot(range(1, 31), cum[:30], 'o', color=C2, ms=6.5, mec='k', mew=0.5)
ax2.plot(range(1, 31), cum[:30], color='gray', alpha=0.3, lw=3, zorder=1)

ax2.set_title('(b)', loc='left', fontweight='normal')
ax2.set_xlabel('$r$')
ax2.set_ylabel(r'$E_r \ [\%]$')

ax2.set_xlim([0, 30])
# ax2.set_ylim([95, 100.5])  # Uncomment if you want to force the standard Y-limits

x_pos = [22, 8, 22, 8]
va_pos = ['top', 'bottom', 'top', 'bottom']

# for ii, (idx, col) in enumerate([(0, 'C3'), (1, 'C0'), (2, 'C5'), (3, 'C8')]):
#     if idx < len(cum):
#         ax2.axhline(cum[idx], color=col, ls=':', lw=1.5)
#         ax2.text(x_pos[ii], cum[idx],
#                  f'$r={idx+1} \\ (E_r={cum[idx]:.1f}\\%)$',
#                  color=col, ha='center', va=va_pos[ii], fontweight='bold',
#                  bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=1))

plt.tight_layout()
#sf('fig5_multiwell_svd_spectrum.pdf')
sf('ch5_17_svd_rank_case5.pdf')
plt.show()

### 5.2 Figure 6 — POD Modes and Temporal Coefficients

In [ ]:
n_show=4
fig, axes = plt.subplots(2,n_show,figsize=(11,4))

mode_phys=['Global depletion\n(all active wells)',
           'W1 vs W3 asymmetry\n+ startup offset',
           'W2 vs flanks\n+ rate changes',
           'Higher-order\ncorrection']

# event times for temporal coefficient colouring
all_events = sorted([t_W2s, t_W3s, t_W1c, t_W2c, t_W3c, t_shut])
event_cols = ['#1a5e9e','#b07318','#0d6e56','#9b2a1a','#6a0dad',"#0FF7E3",'#888888']

period_labels = [
    'PW1 active', 
    'PW2 active', 
    'PW3 active', 
    'PW1 rate change', 
    'PW2 rate change', 
    'PW3 rate change', 
    'Shut-in']

def seg_col(tv):
    if tv < t_W2s:  return event_cols[0]
    elif tv < t_W3s: return event_cols[1]
    elif tv < t_W1c: return event_cols[2]
    elif tv < t_W2c: return event_cols[3]
    elif tv < t_W3c: return event_cols[4]
    elif tv < t_shut: return event_cols[5]
    else: return event_cols[6]

for i in range(n_show):
    ax=axes[0,i]
    ui=U[:,i].reshape(ny,nx)
    vm=np.abs(ui).max()
    im=ax.imshow(ui,origin='lower',cmap='RdBu_r',
                 vmin=-vm,vmax=vm,aspect='auto')
    m_val=f'{sn[i]:.1e}'.split('e')[0]
    e_val=int(f'{sn[i]:.1e}'.split('e')[1])
    for wi,wci in enumerate(wc):
        ax.plot(wci%nx, wci//nx, 'k+', ms=3, mew=0.5, zorder=10)
    ax.set_title(
        f'$u_{{{i+1}}}(\\mathbf{{x}})$\n'
        f'$\\sigma_{{{i+1}}}/\\sigma_1={m_val}\\times10^{{{e_val}}}$\n',fontsize=12)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    plt.colorbar(im,ax=ax,shrink=0.88,format='%.3f').set_label(rf'$u_{i+1}(\mathbf{{x}})\ [-]$', fontsize=12, rotation=90, labelpad=12)

    ax2=axes[1,i]
    ai=sv[i]*Vt[i,:]; ai_n=ai/np.abs(ai).max()
    for j in range(len(t)-1):
        ax2.semilogx(t[j:j+2],ai_n[j:j+2],
                     color=seg_col(t[j]),lw=3,alpha=0.85)
    for tv,col_ in [(t_W2s,WC[1]),(t_W3s,'green'),(t_W1c,'red'),
                     (t_W2c,'purple'),(t_W3c,'cyan'),(t_shut,'gray')]:
        ax2.axvline(tv,color=col_,lw=0.9,ls='--',alpha=0.6)
    ax2.axhline(0,color=GR,lw=0.6,alpha=0.4)
    ax2.set_xlabel('$t$ [hr]',fontsize=12)
    ax2.set_ylabel(f'$a_{i+1}/|a_{i+1}|_{{max}}$',fontsize=12)
    ax2.set_xlim([0.1,1e5]); ax2.set_ylim([-1,1])


    legend_elements = [Patch(facecolor=event_cols[i], label=period_labels[i]) for i in range(7)]


fig.legend(handles=legend_elements, loc='lower center', ncol=7, 
           bbox_to_anchor=(0.5, -0.04), frameon=False, fontsize=11)

plt.tight_layout()
#sf('fig6_multi_rate_pod_modes_temporal.pdf')
sf('ch5_18_pod_modes_case5.pdf')
plt.show()

In [ ]:
GR = '#555555' if 'GR' not in dir() else GR   # fallback if run standalone

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 · UNIFORM LOG-TIME RESAMPLING  (production window, t < t_shut)
# ─────────────────────────────────────────────────────────────────────────────
t_ssa_mask = (t > 0) & (t < t_shut)
t_w1       = t[t_ssa_mask]
dp_w1      = np.clip(dp[0, t_ssa_mask], 1e-6, None)   # ΔP for W1 [psi]

tau      = np.log(t_w1)
tau_uni  = np.linspace(tau.min(), tau.max(), 600)
t_uni    = np.exp(tau_uni)

f_interp = interp1d(tau, dp_w1, kind='cubic')
dp_uni   = f_interp(tau_uni)
dpr_uni  = np.gradient(dp_uni, tau_uni)               # log-domain derivative

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 · HANKEL EMBEDDING
# ─────────────────────────────────────────────────────────────────────────────
def _build_hankel(signal, d):
    n = len(signal)
    H = np.zeros((d, n - d))
    for i in range(d):
        H[i, :] = signal[i : n - d + i]
    return H

d_h    = 20   # embedding dimension
W_h    = 5    # sliding window half-width
k_rank = 3    # number of secondary singular values for Rk

H_dp   = _build_hankel(dp_uni,  d_h)   # from ΔP
H_dpr  = _build_hankel(dpr_uni, d_h)   # from ΔP′

t_Hank = t_uni[d_h:]                   # time axis aligned with Hankel columns

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 · SLIDING SSA — Multi-Rank Complexity & Spectral Entropy
# ─────────────────────────────────────────────────────────────────────────────
def _sliding_ssa(H_mat, t_Hank, W_h, k_rank):
    m      = H_mat.shape[1]
    n_h    = m - W_h
    t_str  = np.zeros(n_h)
    Rk     = np.zeros(n_h)
    H_ent  = np.zeros(n_h)

    for k_s in range(n_h):
        Xw          = H_mat[:, k_s : k_s + W_h]
        t_str[k_s]  = t_Hank[k_s + W_h // 2]
        try:
            _, S_w, _   = np.linalg.svd(Xw, full_matrices=False)
            energies    = S_w ** 2
            E_primary   = energies[0]
            # Multi-Rank Complexity
            Rk[k_s]     = np.sum(energies[1:k_rank]) / E_primary
            # Spectral Entropy
            e_trunc     = energies[energies > 1e-10 * E_primary]
            p_i         = e_trunc / e_trunc.sum()
            H_ent[k_s]  = -np.sum(p_i * np.log(p_i))
        except Exception:
            Rk[k_s]    = np.nan
            H_ent[k_s] = np.nan

    return t_str, Rk, H_ent

t_stream_dp,  Rk_dp,  H_dp_ent  = _sliding_ssa(H_dp,  t_Hank, W_h, k_rank)
t_stream_dpr, Rk_dpr, H_dpr_ent = _sliding_ssa(H_dpr, t_Hank, W_h, k_rank)

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 · SMOOTHING  (Savitzky-Golay on entropy; Rk left raw)
# ─────────────────────────────────────────────────────────────────────────────
def _smooth(arr):
    valid = np.isfinite(arr)
    n     = valid.sum()
    win   = 31 if n > 31 else max(5, (n // 2) * 2 - 1)
    out   = np.full(len(arr), np.nan)
    out[valid] = savgol_filter(arr[valid], window_length=win, polyorder=3)
    return out, valid

H_dp_smooth,  v_dp  = _smooth(H_dp_ent)
H_dpr_smooth, v_dpr = _smooth(H_dpr_ent)

vr_dp  = np.isfinite(Rk_dp)
vr_dpr = np.isfinite(Rk_dpr)

# =============================================================================
# SECTION 5: PLOTTING (With Log-Zoom Inset & Standard Format)
# =============================================================================
# Color palette
C_dp  = '#185FA5'   # blue  — ΔP
C_dpr = '#9b2a1a'   # red   — ΔP'

# Incorporate the Case 5 variable-rate events
ref_events = [
    (t_wbs,  '#2ca02c', r'$t_{\mathrm{wbs}}$ end'),
    (t_W2s,  '#d62728', r'PW2 starts ($q = 800$ STB/d)'),
    (t_int,  '#9467bd', r'$t_{\mathrm{int}}$ -- PW1--PW2 interference'),
    (t_W3s,  '#17becf', r'PW3 starts ($q = 1000$ STB/d)'),
    (t_W1c,  '#1f77b4', r'PW1 rate $\rightarrow$ 400 STB/d'),
    (t_W2c,  '#ff7f0e', r'PW2 rate $\rightarrow$ 500 STB/d'),
    (t_W3c,  '#e377c2', r'PW3 rate $\rightarrow$ 700 STB/d'),
    (t_pss,  '#8c564b', r'$t_{\mathrm{bdf}}$ -- boundary felt'),
    (t_shut, '#7f7f7f', r'All wells shut-in'),
]

fig, axes = plt.subplots(2, 1, figsize=(8, 5.5), sharex=True)
fig.subplots_adjust(hspace=0.25, bottom=0.28) # Extra bottom space for legend

# --- (a) Multi-Rank Complexity Rk — twin y axes ---
ax1      = axes[0]
ax1_twin = ax1.twinx()

ax1.loglog(t_stream_dp[vr_dp],   Rk_dp[vr_dp],   color=C_dp,  lw=3.0, label=r'$R_k$ ($\Delta P$)')
ax1_twin.loglog(t_stream_dpr[vr_dpr], Rk_dpr[vr_dpr], color=C_dpr, lw=3.0, label=r"$R_k$ ($\Delta P'$)", ls='--')

ax1.set_ylabel(r'$R_k$  ($\Delta P$)',  color=C_dp)
ax1_twin.set_ylabel(r"$R_k$  ($\Delta P'$)", color=C_dpr)
ax1.tick_params(axis='y', labelcolor=C_dp)
ax1_twin.tick_params(axis='y', labelcolor=C_dpr)

ax1.set_ylim([1e-10, 1e2])
ax1_twin.set_ylim([1e-10, 1e2])
ax1.set_title('(a)', loc='left')
ax1.xaxis.set_minor_locator(NullLocator())
ax1.yaxis.set_minor_locator(NullLocator())
ax1_twin.yaxis.set_minor_locator(NullLocator())

# Panel (a) inner legend
#_lines1 = ax1.get_lines() + ax1_twin.get_lines()
#ax1.legend(_lines1, [l.get_label() for l in _lines1], fontsize=9, framealpha=0.9, loc='upper left')

# --- (b) Spectral Entropy H — twin y axes ---
ax2      = axes[1]
ax2_twin = ax2.twinx()

ax2.semilogx(t_stream_dp[v_dp],   H_dp_smooth[v_dp],   color=C_dp,  lw=3.0, label=r'$H$ ($\Delta P$)')
ax2_twin.semilogx(t_stream_dpr[v_dpr], H_dpr_smooth[v_dpr], color=C_dpr, lw=3.0, label=r"$H$ ($\Delta P'$)", ls='--')

ax2.set_ylabel(r'$H$  ($\Delta P$)',  color=C_dp)
ax2_twin.set_ylabel(r"$H$  ($\Delta P'$)", color=C_dpr)
ax2.tick_params(axis='y', labelcolor=C_dp)
ax2_twin.tick_params(axis='y', labelcolor=C_dpr)
ax2.set_xlabel(r'$t$  [hours]')
ax2.set_title('(b)', loc='left')

ax2.set_ylim([0, 0.04])
ax2_twin.set_ylim([0, 1.5])
ax2.xaxis.set_minor_locator(NullLocator())
ax2.yaxis.set_minor_locator(NullLocator())
ax2_twin.yaxis.set_minor_locator(NullLocator())

# Panel (b) inner legend
#_lines2 = ax2.get_lines() + ax2_twin.get_lines()
#ax2.legend(_lines2, [l.get_label() for l in _lines2], fontsize=9, framealpha=0.9, loc='upper left')

# =============================================================================
# ── THE INSET FIGURE  ──
# =============================================================================
axins = ax2.inset_axes([0.15, 0.25, 0.4, 0.60])
axins_twin = axins.twinx()

axins.semilogx(t_stream_dp[v_dp], H_dp_smooth[v_dp], color=C_dp, lw=3.0)
axins_twin.semilogx(t_stream_dpr[v_dpr], H_dpr_smooth[v_dpr], color=C_dpr, lw=3.0, ls='--')

for tv, col, _ in ref_events:
    if tv <= 2500:
        axins.axvline(tv, color=col, lw=1.0, ls=':', alpha=0.85)

axins.set_xlim([0.1, 1000])
axins.set_ylim([0, 0.000001])
axins_twin.set_ylim([0, 0.004])

axins.tick_params(axis='both', which='major')
axins.tick_params(axis='y', labelcolor=C_dp)               # Colored inset y-axis (blue)
axins_twin.tick_params(axis='both', which='major')
axins_twin.tick_params(axis='y', labelcolor=C_dpr)         # Colored inset twin y-axis (red)
axins.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.6f'))      
axins_twin.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.4f')) 

bbox_props = dict(facecolor='white', edgecolor='none', alpha=0.8, pad=0.5)
for label in axins.get_yticklabels() + axins_twin.get_yticklabels():
    label.set_bbox(bbox_props)

axins.xaxis.set_minor_locator(NullLocator())
axins.yaxis.set_minor_locator(NullLocator())
axins_twin.yaxis.set_minor_locator(NullLocator())
# =============================================================================

# --- Shared event vertical lines ---
for ax_s in [ax1, ax2]:
    for tv, col, _ in ref_events:
        ax_s.axvline(tv, color=col, lw=1.5, ls=':', alpha=0.85)

ax1.set_xlim([0.1, 1e5])

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8 · LEGEND BOX — dotted coloured lines only, no patches
# ─────────────────────────────────────────────────────────────────────────────
event_handles = [
    Line2D([0], [0], color=col, lw=1.8, ls=':', label=lbl)
    for _, col, lbl in ref_events
]

fig.legend(
    handles        = event_handles,
    loc            = 'lower center',
    ncol           = 3,
    fontsize       = 12.0,
    framealpha     = 1.0,
    edgecolor      = 'none',
    fancybox       = False,
    bbox_to_anchor = (0.5, 0.02),
    title_fontsize = 8.5,
    handlelength   = 1.8,
    handleheight   = 0.9,
    borderpad      = 0.7,
    labelspacing   = 0.35,
)

plt.tight_layout(rect=[0, 0.3, 0.5, 0.5])
sf('ch6_5_ssa_case5.pdf')
plt.show()

---
## 6 · ROM Reconstruction

In [ ]:
r_eq_ft      = 0.2 * P['dx_ft']
kh_well_cell = P['k_well_cell_mD'] * P['h_ft']
peaceman_const = (141.2*P['mu_cp']*P['Bo']/kh_well_cell
                  *(np.log(r_eq_ft/P['rw_ft'])+P['S_skin']))
dp_corr = peaceman_const * q    # (3, m)

ranks  = [3, 6, 12]
colors = ['C0','C5','C9']
styles = ['--','-.', ':']

X_rom = U[:,:r_sel]@np.diag(sv[:r_sel])@Vt[:r_sel,:]+X_mean
eps_g = np.linalg.norm(X-X_rom,'fro')/np.linalg.norm(X,'fro')*100

rom_data={}
t_p1_mask_rom = (t >= 0.1) & (t <= t_W2s)
lnt = np.log(t)
for r in ranks:
    X_r     = U[:,:r]@np.diag(sv[:r])@Vt[:r,:]+X_mean
    bhp_r   = X_r[wc,:] - dp_corr    # (3,m)
    eg_r    = np.linalg.norm(X-X_r,'fro')/np.linalg.norm(X,'fro')*100
    el      = []
    for wi in range(3):
        mask_ = (t>=0.1)&(t<=t_W2s) if wi==0 else (t>=t_W2s)&(t<=t_W1c)
        if mask_.sum()>2:
            el.append(np.linalg.norm(bhp[wi,mask_]-bhp_r[wi,mask_])/
                      np.linalg.norm(bhp[wi,mask_])*100)
        else:
            el.append(np.nan)
    rom_data[r]=dict(bhp_r=bhp_r,eg=eg_r,el=el)

print(f'ROM rank r_sel = {r_sel}  ε_G = {eps_g:.4f}%')
print(f'\n{"r":>4}  {"ε_G":>10}  {"ε_W1 (P1)":>12}  '
      f'{"ε_W2 (P2)":>12}  {"ε_W3 (P3)":>12}')
print('─'*55)
for r in ranks:
    d=rom_data[r]
    vals=[f'{v:>11.3f}%' if np.isfinite(v) else f'{"—":>12}' for v in d['el']]
    print(f'{r:4d}  {d["eg"]:>10.4f}%  '+'  '.join(vals))

### 6.1 Figure 10 — ROM vs Simulator: All Three Wells

In [ ]:
# =============================================================================
# Figure 10 (Modified) — Well 1 POD Comparison (BHP & Derivative)
# Standard matched to Cases 1-4 with Case 5/6 Temporal Color Coding
# =============================================================================

ranks  = [3, 6, 12]
colors = ['magenta', 'purple', 'red']   # magenta replaces orange
styles = ['--', '-.', ':']

t_lo, t_hi = 0.1, 1e5
t_full = (t >= t_lo)
lnt    = np.log(t)

wi = 0   # Well 1

# ── Consistent simulator BHP — Peaceman-corrected snapshot cell pressure ──
bhp_sim_w1 = X[wc[wi], :] - dp_corr[wi, :]        
dp_sim_w1  = P['p_init_psia'] - bhp_sim_w1
dpr_sim_w1 = bourdet(np.clip(dp_sim_w1, 0, None), t, L=0.2)
vs_sim_w1  = t_full & (dp_sim_w1 > 0.1) & (dpr_sim_w1 > 0.1)

# ── Temporal Event Coding (from Multi-Rate Temporal Plot) ─────────────────
event_cols = ['#1a5e9e', '#b07318', '#0d6e56', '#9b2a1a', '#6a0dad', '#0FF7E3', '#888888']
period_labels = [
    'PW1 active', 
    'PW2 active', 
    'PW3 active', 
    'PW1 rate change', 
    'PW2 rate change', 
    'PW3 rate change', 
    'Shut-in'
]

# Time boundaries for the 7 periods
period_times = [t_lo, t_W2s, t_W3s, t_W1c, t_W2c, t_W3c, t_shut, t_hi]

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

# ══════════════════════════════════════════════════════════════════════════
# Panel (a): W1 BHP
# ══════════════════════════════════════════════════════════════════════════
ax = axes[0]

# Simulator line color-coded by region
for i in range(7):
    mask = t_full & (t >= period_times[i]) & (t <= period_times[i+1])
    if mask.any():
        ax.semilogx(t[mask], bhp_sim_w1[mask],
                    color=event_cols[i], alpha=0.95, lw=3.0, zorder=4)

for r, col, ls in zip(ranks, colors, styles):
    d        = rom_data[r]
    bhp_r_w1 = d['bhp_r'][wi, :]
    diff     = bhp_sim_w1[t_full] - bhp_r_w1[t_full]
    el_w     = np.linalg.norm(diff) / np.linalg.norm(bhp_sim_w1[t_full]) * 100
    
    lbl = (rf'POD  $r={r}$  '
           rf'($\varepsilon_G={d["eg"]:.3f}\%$,  '
           rf'$\varepsilon_{{w}}={el_w:.3f}\%$)')
           
    ax.semilogx(t[t_full], bhp_r_w1[t_full],
                color=col, lw=3.0, ls=ls, alpha=0.60, label=lbl)

# Shared vertical boundary times mapped to the event colors
for i in range(1, 7):
    ax.axvline(period_times[i], color=event_cols[i], lw=2.0, ls=':', alpha=0.8)

ax.set_title('(a)', loc='left', weight='normal', fontsize=14)
ax.set_xlabel('$t$ [hours]', fontsize=14)
ax.set_ylabel(r'$p_{wf}$ [psia]', fontsize=14)
ax.set_xlim([t_lo, t_hi])
ax.set_ylim([3800, 4600])
ax.xaxis.set_minor_locator(NullLocator())
ax.yaxis.set_minor_locator(NullLocator())

# ══════════════════════════════════════════════════════════════════════════
# Panel (b): W1 Bourdet Derivative
# ══════════════════════════════════════════════════════════════════════════
ax2 = axes[1]

# Simulator derivative color-coded by region
for i in range(7):
    mask = vs_sim_w1 & (t >= period_times[i]) & (t <= period_times[i+1])
    if mask.any():
        ax2.loglog(t[mask], dpr_sim_w1[mask],
                   color=event_cols[i], alpha=0.95, lw=3.0, zorder=4)

for r, col, ls in zip(ranks, colors, styles):
    d        = rom_data[r]
    bhp_r_w1 = d['bhp_r'][wi, :]
    dp_r     = P['p_init_psia'] - bhp_r_w1
    vf       = t_full & (dp_r > 0.1) & np.isfinite(dp_r)
    spl      = UnivariateSpline(lnt[vf], dp_r[vf], k=5, s=vf.sum() * 0.3)
    dpr_spl  = np.clip(spl.derivative()(lnt), 0, None)
    vs_r     = t_full & (dpr_spl > 0.1)
    
    ax2.loglog(t[vs_r], dpr_spl[vs_r],
               ls=ls, color=col, lw=3.0, alpha=0.60, label=rf'POD  $r={r}$')

# Shared vertical boundary times mapped to the event colors
for i in range(1, 7):
    ax2.axvline(period_times[i], color=event_cols[i], lw=2.0, ls=':', alpha=0.8)

ax2.set_title('(b)', loc='left', weight='normal', fontsize=14)
ax2.set_xlabel('$t$ [hours]', fontsize=14)
ax2.set_ylabel(r"$\Delta p'$ [psi]", fontsize=14)
ax2.set_xlim([t_lo, t_hi])
ax2.set_ylim([0.1, 1000])
ax2.xaxis.set_minor_locator(NullLocator())
ax2.yaxis.set_minor_locator(NullLocator())



# ══════════════════════════════════════════════════════════════════════════
# ── Side-by-Side Dual Legends ─────────────────────────────────────────────

# Extract handles and labels from panel (a) for POD
pod_handles, pod_labels = [], []
for h, l in zip(*ax.get_legend_handles_labels()):
    if 'POD' in l:
        pod_handles.append(h)
        pod_labels.append(l)

# 1. Left Column: Simulator & Temporal Color Patches
sim_handle = mlines.Line2D([], [], color='none', lw=3.0, label='Simulator')
patch_handles = [Patch(facecolor=event_cols[i], alpha=0.8, label=period_labels[i]) for i in range(7)]

# Anchored to the upper right of the x=0.48 midpoint (Left side of center)
# ncol=2 packs the 8 items into 4 tidy rows
leg1 = fig.legend(handles=[sim_handle] + patch_handles, 
                  loc='upper right', bbox_to_anchor=(0.48, 0.02),
                  ncol=2, framealpha=0.0, fontsize=12)

# 2. Right Column: POD Comparison Ranks
# Anchored to the upper left of the x=0.52 midpoint (Right side of center)
leg2 = fig.legend(pod_handles, pod_labels, 
                  loc='upper left', bbox_to_anchor=(0.52, 0.02),
                  ncol=1, framealpha=0.0, fontsize=12)

# Adjust the bottom padding (0.22 to 0.25) depending on how tall the 4-row legend renders
plt.tight_layout(rect=[0.01, 0.03, 1, 1.03])
#sf('fig10_W1_pod_comparison_colored.pdf')
sf('ch5_19_pwf_dp_recon_case5.pdf')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# §  Pressure Field Reconstruction Check — Case 5
#
# Figure A  (ch5_20a_p_field_recon_case5.pdf)
#   n_ranks × 3  —  Ground Truth | POD r | Residual
#   Fixed snapshot at W1 radial-flow window (W1 alone, before W2 startup).
#   Consistent with Cases 1-4 rank-comparison layout.
#
# Figure B  (ch5_20b_p_field_recon_case5.pdf)
#   5_regimes × 3  —  one row per key physical regime, r = r_sel
#   Case-6-specific: captures the full well-startup cascade and buildup.
#   Well locations overlaid as markers on every panel.
# ══════════════════════════════════════════════════════════════════════════

# ── Well pixel positions in the 100×100 imshow frame ─────────────────────
# MRST cartGrid([nx,ny]): flat_idx (0-based) = iy_0*nx + ix_0
# Python reshape(ny,nx) C-order: row = idx//nx, col = idx%nx
# imshow origin='lower': x-axis = col, y-axis = row
wx = [wc[wi_] % nx for wi_ in range(3)]   # column (x) for W1, W2, W3
wy = [wc[wi_] // nx for wi_ in range(3)]  # row    (y) for W1, W2, W3

def _mark_wells(ax_obj):
    """Overlay well markers with WC colour coding."""
    for wi_, (cx, cy) in enumerate(zip(wx, wy)):
        ax_obj.plot(cx, cy, marker='+', ms=3, mew=1,
                    color=WC[wi_], zorder=10)
        # Well text labels removed as requested

def _ki(tv):
    return int(np.argmin(np.abs(t - tv)))

def _cbar(fig_, im_, ax_, label):
    cb = fig_.colorbar(im_, ax=ax_, shrink=0.70, pad=0.03)
    cb.set_label(label, size=12)
    cb.ax.tick_params(labelsize=12)

# ── Representative snapshot times ─────────────────────────────────────────
# Fig A reference: W1 radial-flow midpoint (W1 alone, pre-W2)
t_rf_snap = np.sqrt(t_wbs * t_W2s)
ki_rf     = _ki(t_rf_snap)

# Fig B: five key regimes
t_w1w2_snap = np.sqrt(t_W2s  * t_W3s)    # W1+W2 active, pre-W3
t_all3_snap = np.sqrt(t_W3s  * t_W1c)    # all 3 wells, pre-rate-change
t_bdf_snap  = np.sqrt(t_pss  * t_shut)   # BDF (boundary-dominated)
t_bu_snap   = t_shut + 0.20 * (t[-1] - t_shut)  # early buildup

regime_snaps = [
    (_ki(t_rf_snap),   rf'W1 alone  $t={t[_ki(t_rf_snap)]:.0f}$ hr',
     WC[0]),
    (_ki(t_w1w2_snap), rf'W1+W2  $t={t[_ki(t_w1w2_snap)]:.0f}$ hr',
     WC[1]),
    (_ki(t_all3_snap), rf'All 3 wells  $t={t[_ki(t_all3_snap)]:.0f}$ hr',
     '#6a0dad'),
    (_ki(t_bdf_snap),  rf'BDF  $t={t[_ki(t_bdf_snap)]:.0f}$ hr',
     GR),
    (_ki(t_bu_snap),   rf'Buildup  $t={t[_ki(t_bu_snap)]:.0f}$ hr',
     WC[2]),
]

print('Snapshot times:')
for ki_, lbl_, _ in regime_snaps:
    print(f'  {lbl_}')

# ══════════════════════════════════════════════════════════════════════════
# FIGURE A — rank comparison at W1 RF snapshot (consistent with Cases 1-4)
# ══════════════════════════════════════════════════════════════════════════
ranks   = [3, 6, 12]
n_ranks = len(ranks)

ki    = ki_rf+8
F_gt  = dX[:, ki].reshape(ny, nx)
vm    = max(F_gt.max(), 0.1)

fig_a, axes_a = plt.subplots(n_ranks, 3, figsize=(8, 6))

for ri, r in enumerate(ranks):
    X_r      = U[:, :r] @ np.diag(sv[:r]) @ Vt[:r, :] + X_mean
    dX_r     = (P['p_init_psia'] - X_r)[:, ki].reshape(ny, nx)
    diff     = F_gt - dX_r
    eps_step = (np.linalg.norm(X[:, ki] - X_r[:, ki]) /
                np.linalg.norm(X[:, ki]) * 100)
    vd       = max(np.abs(diff).max(), 0.01)

    ax_gt  = axes_a[ri, 0]
    ax_pod = axes_a[ri, 1]
    ax_res = axes_a[ri, 2]

    # Ground truth — top-left only
    if ri == 0:
        im1 = ax_gt.imshow(F_gt, origin='lower', cmap='hot_r',
                           vmin=0, vmax=vm)
        ax_gt.set_title(rf'Ground Truth' + '\n'
                        + rf'$t={t[ki]:.0f}$ hr  (W1 RF)',
                        fontweight='normal', fontsize=12)
        _cbar(fig_a, im1, ax_gt, r'$\Delta p$ [psi]')
        _mark_wells(ax_gt)
    else:
        ax_gt.axis('off')

    im2 = ax_pod.imshow(dX_r, origin='lower', cmap='hot_r',
                        vmin=0, vmax=vm)
    ax_pod.set_title(rf'POD ($r={r}$)' + '\n'
                     + rf'$\varepsilon_G={eps_step:.4f}\%$',
                     fontweight='normal', fontsize=12)
    _cbar(fig_a, im2, ax_pod, r'$\Delta p$ [psi]')
    _mark_wells(ax_pod)

    im3 = ax_res.imshow(diff, origin='lower', cmap='seismic',
                        vmin=-vd, vmax=vd)
    ax_res.set_title(rf'Residual ($r={r}$)' + '\n'
                     + rf'Max $\Delta$: {vd:.2f} psi',
                     fontweight='normal', fontsize=12)
    _cbar(fig_a, im3, ax_res, r'$\Delta p$ [psi]')
    _mark_wells(ax_res)

    for ax_obj in [ax_gt, ax_pod, ax_res]:
        if ax_obj.axison:
            ax_obj.set_xticks([]); ax_obj.set_yticks([])

fig_a.tight_layout()
sf('ch5_20a_p_field_recon_case5.pdf')
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# FIGURE B — multi-regime reconstruction at r_sel (case-6-specific)
# ══════════════════════════════════════════════════════════════════════════
X_rsel = U[:, :r_sel] @ np.diag(sv[:r_sel]) @ Vt[:r_sel, :] + X_mean
n_reg  = len(regime_snaps)

fig_b, axes_b = plt.subplots(n_reg, 3, figsize=(8, 2.0 * n_reg))

for pi, (ki, row_lbl, col) in enumerate(regime_snaps):
    F_gt     = dX[:, ki].reshape(ny, nx)
    dX_r     = (P['p_init_psia'] - X_rsel)[:, ki].reshape(ny, nx)
    diff     = F_gt - dX_r
    eps_step = (np.linalg.norm(X[:, ki] - X_rsel[:, ki]) /
                np.linalg.norm(X[:, ki]) * 100)
    vd       = max(np.abs(diff).max(), 0.01)
    vm_p     = max(F_gt.max(), 0.1)   # per-regime vmax (field evolves strongly)

    ax_gt  = axes_b[pi, 0]
    ax_pod = axes_b[pi, 1]
    ax_res = axes_b[pi, 2]

    im1 = ax_gt.imshow(F_gt, origin='lower', cmap='hot_r',
                       vmin=0, vmax=vm_p)
    ax_gt.set_title(row_lbl + '\nGround Truth',
                    fontweight='normal', fontsize=12, color=col)
    _cbar(fig_b, im1, ax_gt, r'$\Delta p$ [psi]')
    _mark_wells(ax_gt)

    im2 = ax_pod.imshow(dX_r, origin='lower', cmap='hot_r',
                        vmin=0, vmax=vm_p)
    ax_pod.set_title(row_lbl + rf'  POD $r={r_sel}$' + '\n'
                     + rf'$\varepsilon_G={eps_step:.4f}\%$',
                     fontweight='normal', fontsize=12, color=col)
    _cbar(fig_b, im2, ax_pod, r'$\Delta p$ [psi]')
    _mark_wells(ax_pod)

    im3 = ax_res.imshow(diff, origin='lower', cmap='seismic',
                        vmin=-vd, vmax=vd)
    ax_res.set_title(row_lbl + '\nResidual  '
                     + rf'Max $\Delta$: {vd:.2f} psi',
                     fontweight='normal', fontsize=12, color=col)
    _cbar(fig_b, im3, ax_res, r'$\Delta p$ [psi]')
    _mark_wells(ax_res)

    for ax_obj in [ax_gt, ax_pod, ax_res]:
        ax_obj.set_xticks([]); ax_obj.set_yticks([])

fig_b.tight_layout()
sf('ch5_20b_p_field_recon_case5.pdf')
plt.show()

---
## 7 · Summary

In [ ]:
import json as js
summary=dict(
    case='case5_multiwell_staggered',
    n_wells=3, n_periods=7,
    k_mD=float(k), kh=float(kh),
    startup_times=[float(t_W1s),float(t_W2s),float(t_W3s)],
    change_times=[float(t_W1c),float(t_W2c),float(t_W3c)],
    t_int=float(t_int), t_bdf=float(t_pss),
    r_sel=int(r_sel), r999=int(r999),
    sigma1=float(sv[0]), sigma2_ratio=float(sn[1]),
    eps_global=float(eps_g),

)
out_file=PROJECT/'data'/'processed'/'case5_summary.json'
with open(out_file,'w') as f: js.dump(summary,f,indent=2)
print('── SUMMARY ──────────────────────────────────')
for k_,v_ in summary.items(): print(f'  {k_:<28}: {v_}')
print(f'Saved: {out_file}')